# 00 Colab Bootstrap

Hand-maintained revision runbook. Do not regenerate from `scripts/revision/create_revision_notebooks.py` unless that generator is first updated to the current revision protocol.


## Purpose

Use this first when opening Colab directly. It mounts Drive, checks large artifacts, clones or updates the GitHub branch used as the code transport layer, and prepares the Mamba/Torch runtime required by Notebook 02a.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = 'main'
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)

print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)


## Verify Drive Artifacts

In [ ]:
required_files = [
    'WFDB-ChapmanShaoxing.zip',
    'PTB-XL.zip',
    'Georgia.zip',
    'cpsc2021.zip',
]

missing = []
for name in required_files:
    path = DRIVE_ROOT / name
    print(f'{name}: exists={path.exists()} path={path}')
    if not path.exists():
        missing.append(str(path))

model_dir = DRIVE_ROOT / 'model'
legacy_fold_ckpts = sorted(model_dir.glob('fold*_best.pt'))
explicit_fold_ckpts = sorted(model_dir.glob('fold*_final_ema.pt'))
print('\nmodel_dir       :', model_dir)
print('legacy best checkpoints     :', len(legacy_fold_ckpts))
print('explicit final EMA checkpoints:', len(explicit_fold_ckpts))
print('fold PCA manifest:', Path('reports/revision/manifests/fold_pca_manifest.json').exists())

if len(legacy_fold_ckpts) < 5 and len(explicit_fold_ckpts) < 5:
    print('Checkpoint note: no complete five-fold set is present yet. This is expected before Notebook 02a retraining.')
if missing:
    raise FileNotFoundError('Missing required Drive artifacts:\n' + '\n'.join(missing))


## Clone Or Update Repo

In [ ]:
def run(cmd, cwd=None):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=True)

if (REPO_DIR / '.git').exists():
    run('git fetch origin', cwd=REPO_DIR)
    run(f'git checkout {BRANCH}', cwd=REPO_DIR)
    run(f'git pull --ff-only origin {BRANCH}', cwd=REPO_DIR)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repo. Rename it or set REPO_DIR to another folder.')
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
run('git status --short --branch')


## Install Lightweight Dependencies

In [ ]:
!python --version
!pip install -q "numpy>=2.0,<2.1" "scipy>=1.14.1,<2.0" "pandas==2.2.2" "scikit-learn>=1.2,<1.9" "requests==2.32.4" "pillow>=8,<12" tqdm "wfdb==4.1.2" joblib matplotlib seaborn packaging neurokit2 iterative-stratification thop


## Verify GPU Runtime

In [ ]:
import sys

try:
    import torch
except Exception as exc:
    raise RuntimeError('PyTorch is not importable after base dependency setup. Restart the runtime and rerun Notebook 00.') from exc

print('Python:', sys.version.split()[0])
print('Torch :', torch.__version__, 'CUDA:', torch.version.cuda or 'CPU', 'available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'This Colab runtime is CPU-only, but ECG-RAMBA Mamba training/inference requires CUDA-enabled PyTorch. '
        'In Colab select Runtime > Change runtime type > Hardware accelerator > GPU, preferably A100 High-RAM; '
        'then disconnect/delete this CPU runtime and rerun Notebook 00 from the first cell.'
    )


## Install Model Dependencies / Mamba Runtime

In [ ]:
import json
from pathlib import Path

installer_nb = json.loads((REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb').read_text(encoding='utf-8'))
installer_source = None
for cell in installer_nb['cells']:
    if cell.get('cell_type') != 'code':
        continue
    source = ''.join(cell.get('source', []))
    if 'INSTALL_MODEL_DEPS = True' in source and 'AUTO_PIN_TORCH_FOR_MAMBA' in source:
        installer_source = source
        break
if installer_source is None:
    raise RuntimeError('Could not locate the canonical Mamba installer cell in notebook 02.')
print('Running canonical Mamba installer from Notebook 02...')
exec(compile(installer_source, str(REPO_DIR / 'notebooks/02_predictions_and_external_eval.ipynb:cell7'), 'exec'), globals(), globals())


## Run A0 Audit

In [ ]:
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.chdir(REPO_DIR)
!python scripts/revision/00_audit_protocol.py
